# gl-yolo — sweep complet (MVA → DOTA → VisDrone)

Entraîne les 8 architectures sur les 3 datasets (24 runs), 200 epochs, imgsz=1280.

**Avant de lancer** : Runtime → Change runtime type → GPU (A100 si Colab Pro+ le propose).

**Ce run va prendre plusieurs jours.** Le repo est cloné *dans Drive* (pas sur le disque local éphémère) pour que `runs/` et `logs/` survivent aux déconnexions de session : à chaque reconnexion, ré-exécutez les cellules dans l'ordre — `run_gpu.py` saute automatiquement les modèles déjà terminés et reprend le modèle interrompu depuis son dernier checkpoint au lieu de le refaire depuis zéro. Les datasets, eux, sont automatiquement téléchargés sur le disque local (pas Drive) par `train.py` lui-même -- pas de cellule à se souvenir de ré-exécuter pour ça.

## 1. Vérifier le GPU attribué

In [ ]:
!nvidia-smi
import torch
print("CUDA disponible:", torch.cuda.is_available())

## 2. Monter Google Drive (persistance entre sessions)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Cloner (1re fois) ou mettre à jour le repo sur Drive

In [ ]:
import os

REPO_DIR = "/content/drive/MyDrive/gl-yolo"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/osiastossou/gl-yolo.git "{REPO_DIR}"
else:
    print("Repo déjà présent sur Drive, mise à jour...")
    !git -C "{REPO_DIR}" pull

%cd {REPO_DIR}

## 4. Installer les dépendances

In [ ]:
!pip install -q -e .

## 5. (Optionnel) vérifier l'état d'un sweep précédent

Utile après une reconnexion, pour voir ce qui est déjà fini avant de relancer.

In [ ]:
!find runs -name best.pt 2>/dev/null
!find runs -name last.pt 2>/dev/null

## 6. Lancer le sweep complet — depuis le **Terminal**, pas une cellule

MVA (bird, single-class) en premier, puis DOTA, puis VisDrone — 8 architectures chacun.

**Important** : un process lancé en arrière-plan depuis une cellule du notebook (`subprocess.Popen`, `!cmd &`, ...) est classé par Colab dans le cgroup `jupyter-children`, qui a une limite RAM bien plus basse que la RAM totale affichée -- un entraînement YOLO la dépasse vite et se fait tuer (`Memory cgroup out of memory`, code -9), peu importe la taille du modèle. Un Terminal (Pro/Pro+, icône dans la barre latérale gauche ou Tools → Terminal) tourne comme un process indépendant, hors de ce cgroup restreint.

Dans le **Terminal** (pas ici) :
```bash
cd /content/drive/MyDrive/gl-yolo
nohup python run_gpu.py -epochs 200 -imgsz 1280 > sweep_colab.log 2>&1 &
disown
```
Le notebook ne sert plus qu'à surveiller (section 7) -- il ne lance plus rien lui-même.

## 7. Suivre la progression

Ré-exécutez cette cellule à tout moment pour voir : quel job tourne, l'epoch en cours et le détail du batch courant, et les métriques déjà sauvegardées pour ce job. Elle ne fait que lire des fichiers écrits par le process du Terminal -- aucun risque de la relancer.

In [ ]:
import glob
import os

print("--- Sweep : dernier job démarré/terminé ---")
!tail -5 sweep_colab.log 2>/dev/null

log_files = glob.glob("logs/*.log")
if not log_files:
    print("\nPas encore de log -- le premier job n'a pas encore démarré.")
else:
    latest = max(log_files, key=os.path.getmtime)
    name = os.path.splitext(os.path.basename(latest))[0]
    print(f"\n--- Job en cours : {name} (epoch/batch courant) ---")
    # le log contient des retours chariot (\r) pour la barre de progression -> on les
    # convertit en vrais saut de lignes pour voir le dernier état proprement
    !tail -c 4000 "{latest}" | tr '\r' '\n' | tail -5

    csv_path = f"runs/{name}/results.csv"
    if os.path.exists(csv_path):
        import pandas as pd

        print(f"\n--- Epochs déjà sauvegardés ({csv_path}) ---")
        print(pd.read_csv(csv_path).tail(5).to_string(index=False))
    else:
        print("\nPas encore de checkpoint sauvegardé pour ce job (results.csv pas encore créé).")

## En cas de déconnexion

Reconnectez-vous, ré-exécutez les cellules **2 à 6** dans l'ordre (Drive → repo → deps → sweep). `run_gpu.py` saute les modèles déjà terminés (`best.pt` présent) et reprend automatiquement celui qui était en cours (`last.pt` sans `best.pt`) depuis son dernier checkpoint. Le dossier dataset se re-télécharge en local à chaque nouvelle session (quelques minutes) -- normal, c'est voulu.